# Hex Base Game

In [1]:
import numpy as np

class HexState:
    def __init__(self, board_size=11, board=None, turn=1):
        """
        Represents the state of a Hex game.
        
        Player 1 (Vertical): Tries to connect top row (row 0) to bottom row (size-1).
        Player 2 (Horizontal): Tries to connect left column (col 0) to right column (size-1).
        """
        self.board_size = board_size
        self.turn = turn  # 1 or 2
        
        if board is None:
            self.board = np.zeros((board_size, board_size), dtype=int)
        else:
            self.board = np.copy(board)
            
        self._cached_winner = None
        self._is_terminal_cached = False

    def get_legal_moves(self):
        """Returns a list of tuples (row, col) representing empty spaces."""
        if self.is_terminal():
            return []
        # Non-zero entries are occupied
        rows, cols = np.where(self.board == 0)
        return list(zip(rows, cols))

    def make_move(self, move):
        """
        Returns a NEW HexState instance representing the state after the move.
        Crucial for MCTS to avoid modifying historical board states during simulation.
        """
        row, col = move
        if self.board[row, col] != 0:
            raise ValueError(f"Move {move} is invalid. Cell already occupied.")
            
        next_board = np.copy(self.board)
        next_board[row, col] = self.turn
        next_turn = 3 - self.turn  # Swaps between 1 and 2
        
        return HexState(self.board_size, next_board, next_turn)

    def get_neighbors(self, row, col):
        """Returns the up to 6 valid neighboring coordinates on a Hex grid."""
        neighbors = []
        # The 6 standard hex directions
        directions = [
            (row - 1, col), (row - 1, col + 1),
            (row, col - 1), (row, col + 1),
            (row + 1, col - 1), (row + 1, col)
        ]
        for r, c in directions:
            if 0 <= r < self.board_size and 0 <= c < self.board_size:
                neighbors.append((r, c))
        return neighbors

    def check_winner(self):
        """
        Determines the winner of the game using an efficient BFS pathfinder.
        Hex cannot end in a draw, so if the board is full, one player must have won.
        """
        if self._cached_winner is not None:
            return self._cached_winner

        # Check Player 1: Top to Bottom connection
        if self._has_connected_path(player=1):
            self._cached_winner = 1
            return 1
            
        # Check Player 2: Left to Right connection
        if self._has_connected_path(player=2):
            self._cached_winner = 2
            return 2

        return None

    def _has_connected_path(self, player):
        """BFS helper to check connectivity from border to border for a given player."""
        queue = []
        visited = set()

        if player == 1:
            # Player 1 starts at any cell in the top row (row 0) owned by Player 1
            for c in range(self.board_size):
                if self.board[0, c] == 1:
                    queue.append((0, c))
                    visited.add((0, c))
        else:
            # Player 2 starts at any cell in the left column (col 0) owned by Player 2
            for r in range(self.board_size):
                if self.board[r, 0] == 2:
                    queue.append((r, 0))
                    visited.add((r, 0))

        while queue:
            curr_r, curr_c = queue.pop(0)

            # Check target boundary conditions
            if player == 1 and curr_r == self.board_size - 1:
                return True
            if player == 2 and curr_c == self.board_size - 1:
                return True

            for nr, nc in self.get_neighbors(curr_r, curr_c):
                if self.board[nr, nc] == player and (nr, nc) not in visited:
                    visited.add((nr, nc))
                    queue.append((nr, nc))

        return False

    def is_terminal(self):
        """Returns True if a player has won or if the board is full."""
        if self._is_terminal_cached:
            return True
            
        winner = self.check_winner()
        if winner is not None:
            self._is_terminal_cached = True
            return True
            
        # If no empty slots remain, the game is terminal
        if not np.any(self.board == 0):
            self._is_terminal_cached = True
            return True
            
        return False

    def __str__(self):
        """Generates a text-based representation of the rhomboid Hex board."""
        symbols = {0: '.', 1: 'X', 2: 'O'}
        lines = []
        for r in range(self.board_size):
            indent = " " * r
            row_str = " ".join(symbols[self.board[r, c]] for c in range(self.board_size))
            lines.append(f"{indent}{row_str}")
        return "\n".join(lines)

# Running the Base Hex Game

In [2]:
if __name__ == "__main__":
    # 1. Initialize a 4x4 test environment to keep validation execution swift
    print("Initializing a 4x4 Hex Board...")
    state = HexState(board_size=4)
    print(state)
    print("-" * 20)

    # 2. Simulate random play until a terminal state is reached
    import random
    
    step = 1
    while not state.is_terminal():
        moves = state.get_legal_moves()
        chosen_move = random.choice(moves)
        
        print(f"Step {step}: Player {state.turn} plays {chosen_move}")
        state = state.make_move(chosen_move)
        print(state)
        print("-" * 20)
        step += 1

    # 3. Output results
    winner = state.check_winner()
    print(f"Game Over! Winner: Player {winner} ({'X' if winner == 1 else 'O'})")

Initializing a 4x4 Hex Board...
. . . .
 . . . .
  . . . .
   . . . .
--------------------
Step 1: Player 1 plays (np.int64(3), np.int64(2))
. . . .
 . . . .
  . . . .
   . . X .
--------------------
Step 2: Player 2 plays (np.int64(0), np.int64(1))
. O . .
 . . . .
  . . . .
   . . X .
--------------------
Step 3: Player 1 plays (np.int64(3), np.int64(1))
. O . .
 . . . .
  . . . .
   . X X .
--------------------
Step 4: Player 2 plays (np.int64(1), np.int64(2))
. O . .
 . . O .
  . . . .
   . X X .
--------------------
Step 5: Player 1 plays (np.int64(0), np.int64(0))
X O . .
 . . O .
  . . . .
   . X X .
--------------------
Step 6: Player 2 plays (np.int64(1), np.int64(1))
X O . .
 . O O .
  . . . .
   . X X .
--------------------
Step 7: Player 1 plays (np.int64(3), np.int64(0))
X O . .
 . O O .
  . . . .
   X X X .
--------------------
Step 8: Player 2 plays (np.int64(1), np.int64(0))
X O . .
 O O O .
  . . . .
   X X X .
--------------------
Step 9: Player 1 plays (np.int64(1), 

# Base Gomoku Game

In [3]:
import numpy as np

class GomokuState:
    def __init__(self, board_size=15, board=None, turn=1, last_move=None):
        """
        Represents the state of a Gomoku game.
        
        Player 1: Plays 'X' (Black) - usually goes first.
        Player 2: Plays 'O' (White).
        Win Condition: Exactly or at least 5 consecutive pieces in a row 
                       (horizontal, vertical, or diagonal).
        """
        self.board_size = board_size
        self.turn = turn  # 1 or 2
        self.last_move = last_move  # Tracks the last played (row, col) for optimized win checking
        
        if board is None:
            self.board = np.zeros((board_size, board_size), dtype=int)
        else:
            self.board = np.copy(board)
            
        self._cached_winner = None
        self._is_terminal_cached = False

    def get_legal_moves(self):
        """Returns a list of tuples (row, col) representing empty spaces."""
        if self.is_terminal():
            return []
        rows, cols = np.where(self.board == 0)
        return list(zip(rows, cols))

    def make_move(self, move):
        """
        Returns a NEW GomokuState instance representing the state after the move.
        Ensures thread-safe/branch-safe execution during MCTS tree expansions.
        """
        row, col = move
        if self.board[row, col] != 0:
            raise ValueError(f"Move {move} is invalid. Cell already occupied.")
            
        next_board = np.copy(self.board)
        next_board[row, col] = self.turn
        next_turn = 3 - self.turn  # Swaps between 1 and 2
        
        return GomokuState(self.board_size, next_board, next_turn, last_move=move)

    def check_winner(self):
        """
        Determines the winner of the game.
        Optimized to scan outwards from the last played move across 4 directional axes.
        """
        if self._cached_winner is not None:
            return self._cached_winner

        if self.last_move is None:
            return None

        r, c = self.last_move
        player = self.board[r, c]  # The player who just made the move

        # The 4 directions to check: Horizontal, Vertical, Diagonal Down-Right, Diagonal Up-Right
        directions = [(0, 1), (1, 0), (1, 1), (1, -1)]

        for dr, dc in directions:
            count = 1  # Include the piece just placed
            
            # Search forward along the direction vector
            step = 1
            while True:
                nr, nc = r + dr * step, c + dc * step
                if 0 <= nr < self.board_size and 0 <= nc < self.board_size and self.board[nr, nc] == player:
                    count += 1
                    step += 1
                else:
                    break
                    
            # Search backward along the direction vector
            step = 1
            while True:
                nr, nc = r - dr * step, c - dc * step
                if 0 <= nr < self.board_size and 0 <= nc < self.board_size and self.board[nr, nc] == player:
                    count += 1
                    step += 1
                else:
                    break

            # Gomoku Win Check (Standard rule: 5 or more in a row wins)
            if count >= 5:
                self._cached_winner = player
                return player

        return None

    def is_terminal(self):
        """Returns True if a player has achieved 5-in-a-row or if the board is completely full (Draw)."""
        if self._is_terminal_cached:
            return True
            
        winner = self.check_winner()
        if winner is not None:
            self._is_terminal_cached = True
            return True
            
        # If no empty slots remain, it's a draw terminal state
        if not np.any(self.board == 0):
            self._is_terminal_cached = True
            return True
            
        return False

    def __str__(self):
        """Generates a text-based grid representation of the Gomoku board."""
        symbols = {0: '.', 1: 'X', 2: 'O'}
        lines = []
        
        # Add column headers for easier debugging
        header = "   " + " ".join(f"{c:02d}"[-2:] for c in range(self.board_size))
        lines.append(header)
        
        for r in range(self.board_size):
            row_str = " ".join(symbols[self.board[r, c]] for c in range(self.board_size))
            lines.append(f"{r:02d} {row_str}")
            
        return "\n".join(lines)

# Running the Base Gomoku Game

In [4]:
if __name__ == "__main__":
    # 1. Initialize a 6x6 test environment for fast evaluation
    print("Initializing a 6x6 Gomoku Board...")
    state = GomokuState(board_size=6)
    print(state)
    print("-" * 30)

    # 2. Simulate random play until a terminal state is reached
    import random
    
    step = 1
    while not state.is_terminal():
        moves = state.get_legal_moves()
        chosen_move = random.choice(moves)
        
        print(f"Step {step}: Player {state.turn} plays {chosen_move}")
        state = state.make_move(chosen_move)
        print(state)
        print("-" * 30)
        step += 1

    # 3. Output results
    winner = state.check_winner()
    if winner:
        print(f"Game Over! Winner: Player {winner} ({'X' if winner == 1 else 'O'})")
    else:
        print("Game Over! It's a Draw.")

Initializing a 6x6 Gomoku Board...
   00 01 02 03 04 05
00 . . . . . .
01 . . . . . .
02 . . . . . .
03 . . . . . .
04 . . . . . .
05 . . . . . .
------------------------------
Step 1: Player 1 plays (np.int64(3), np.int64(1))
   00 01 02 03 04 05
00 . . . . . .
01 . . . . . .
02 . . . . . .
03 . X . . . .
04 . . . . . .
05 . . . . . .
------------------------------
Step 2: Player 2 plays (np.int64(0), np.int64(4))
   00 01 02 03 04 05
00 . . . . O .
01 . . . . . .
02 . . . . . .
03 . X . . . .
04 . . . . . .
05 . . . . . .
------------------------------
Step 3: Player 1 plays (np.int64(5), np.int64(5))
   00 01 02 03 04 05
00 . . . . O .
01 . . . . . .
02 . . . . . .
03 . X . . . .
04 . . . . . .
05 . . . . . X
------------------------------
Step 4: Player 2 plays (np.int64(5), np.int64(0))
   00 01 02 03 04 05
00 . . . . O .
01 . . . . . .
02 . . . . . .
03 . X . . . .
04 . . . . . .
05 O . . . . X
------------------------------
Step 5: Player 1 plays (np.int64(0), np.int64(0))
   00

# The Core Node Structure

In [5]:
import numpy as np

class MCTSNode:
    def __init__(self, state, parent=None, move_from_parent=None):
        """
        A node in the Monte Carlo Tree.
        Works for both Hex and Gomoku as long as they share the same API.
        """
        self.state = state
        self.parent = parent
        self.move_from_parent = move_from_parent
        self.children = {}  # Format: {move: MCTSNode}
        
        # Core statistics used by both UCB and Thompson Sampling
        self.visit_count = 0
        self.win_count = 0  
        
        # Unexplored actions from this specific state
        self.untried_moves = state.get_legal_moves()
        
    def is_fully_expanded(self):
        """A node is fully expanded if all legal moves have been explored."""
        return len(self.untried_moves) == 0
        
    def is_terminal_node(self):
        """Checks if this node represents a finished game."""
        return self.state.is_terminal()
        
    def expand(self):
        """
        Pops an untried move, creates a new child state, and adds it to the tree.
        """
        # Take the last move from the untried list (O(1) operation)
        move = self.untried_moves.pop()
        
        # Generate the new state using your immutable make_move function
        next_state = self.state.make_move(move)
        
        # Create and link the new child node
        child_node = MCTSNode(next_state, parent=self, move_from_parent=move)
        self.children[move] = child_node
        
        return child_node
        
    def backpropagate(self, result):
        """
        Updates this node's statistics and recursively passes the result up the tree.
        
        result: 1 (Player 1 wins), 2 (Player 2 wins), or None (Draw)
        """
        self.visit_count += 1
        
        # MCTS perspective tracking: 
        # 'self.state.turn' tells us who is ABOUT to move. 
        # Therefore, the player who made the move to GET to this node is (3 - self.state.turn).
        player_who_just_moved = 3 - self.state.turn
        
        if result == player_who_just_moved:
            self.win_count += 1
        elif result is None:
            # For Gomoku draws, award a half-point
            self.win_count += 0.5 
            
        # Pass the result up to the root
        if self.parent is not None:
            self.parent.backpropagate(result)

# The Selection Algorithms

## Algorithm A: Standard UCT

In [6]:
def select_child_uct(node, exploration_constant=1.414):
    """
    Selects the best child using the Upper Confidence Bound applied to Trees (UCT).
    """
    best_score = -float('inf')
    best_child = None
    
    for child in node.children.values():
        # Exploit: The current win rate
        exploit_term = child.win_count / child.visit_count
        
        # Explore: Favors nodes with fewer visits
        explore_term = exploration_constant * np.sqrt(np.log(node.visit_count) / child.visit_count)
        
        ucb_score = exploit_term + explore_term
        
        if ucb_score > best_score:
            best_score = ucb_score
            best_child = child
            
    return best_child

## Thompson Sampling MCTS

In [7]:
def select_child_thompson(node, prior_alpha=1.0, prior_beta=1.0):
    """
    Selects the best child using Thompson Sampling.
    Models the win probability of each child as a Beta distribution.
    """
    best_sample = -float('inf')
    best_child = None
    
    for child in node.children.values():
        # Calculate the posterior parameters based on the node's history
        # alpha = prior + wins
        # beta = prior + losses (visits - wins)
        alpha = prior_alpha + child.win_count
        beta = prior_beta + (child.visit_count - child.win_count)
        
        # Draw a sample from the Beta distribution
        sampled_value = np.random.beta(alpha, beta)
        
        if sampled_value > best_sample:
            best_sample = sampled_value
            best_child = child
            
    return best_child

# The Rollout Policy & Main Loop

In [8]:
import random
import time

def random_rollout(state):
    """
    Phase 3: Simulation.
    Plays random moves from the current state until a terminal condition is met.
    Returns the winner (1 or 2, or None for a draw).
    """
    current_state = state
    while not current_state.is_terminal():
        legal_moves = current_state.get_legal_moves()
        # Fast random choice
        move = random.choice(legal_moves)
        current_state = current_state.make_move(move)
    return current_state.check_winner()


def run_mcts(root_state, iterations, selection_strategy, **strategy_kwargs):
    """
    Executes the MCTS algorithm for a fixed number of iterations.
    
    selection_strategy: Either select_child_uct or select_child_thompson
    strategy_kwargs: Hyperparameters like exploration_constant or priors
    """
    root_node = MCTSNode(state=root_state)

    for _ in range(iterations):
        node = root_node

        # Phase 1: Selection
        # Traverse down the tree using the chosen selection strategy 
        # until we hit a node that isn't fully expanded or is terminal.
        while node.is_fully_expanded() and not node.is_terminal_node():
            node = selection_strategy(node, **strategy_kwargs)

        # Phase 2: Expansion
        # If the node is not terminal, expand it by creating one new child.
        if not node.is_terminal_node():
            node = node.expand()

        # Phase 3: Simulation (Rollout)
        # Play out the game randomly from this new node state.
        result = random_rollout(node.state)

        # Phase 4: Backpropagation
        # Feed the result back up to the root node.
        node.backpropagate(result)

    # After finishing iterations, choose the final move.
    # In MCTS, the best move is the one belonging to the most visited child.
    best_move = max(root_node.children.items(), key=lambda item: item[1].visit_count)[0]
    
    return best_move, root_node

# Setting Up The Instrumentation (Metrics Collection)

In [9]:
def calculate_root_entropy(root_node):
    """
    Calculates the Shannon Entropy of the root node's child visit distribution.
    Higher entropy = Thompson Sampling explored more broadly / diversely.
    Lower entropy = UCT focused heavily on a single path.
    """
    if not root_node.children:
        return 0.0
        
    total_visits = sum(child.visit_count for child in root_node.children.values())
    if total_visits == 0:
        return 0.0
        
    entropy = 0.0
    for child in root_node.children.values():
        if child.visit_count > 0:
            p_i = child.visit_count / total_visits
            entropy -= p_i * np.log(p_i)
            
    return entropy

# The Ultimate Test Drive

In [10]:
if __name__ == "__main__":
    # Initialize a small Hex board for an instant sanity check
    game_state = HexState(board_size=4)
    print("--- Starting AI vs AI Test Match ---")
    print(game_state)
    print("-" * 40)
    
    while not game_state.is_terminal():
        start_time = time.time()
        
        if game_state.turn == 1:
            # Player 1 uses UCT
            print("UCT (Player 1) is thinking...")
            move, root = run_mcts(game_state, iterations=400, selection_strategy=select_child_uct, exploration_constant=1.4)
            strategy_used = "UCT"
        else:
            # Player 2 uses Thompson Sampling
            print("Thompson Sampling (Player 2) is thinking...")
            move, root = run_mcts(game_state, iterations=400, selection_strategy=select_child_thompson, prior_alpha=1.0, prior_beta=1.0)
            strategy_used = "Thompson"
            
        elapsed = time.time() - start_time
        entropy = calculate_root_entropy(root)
        
        # Execute move
        game_state = game_state.make_move(move)
        
        print(f"[{strategy_used}] Selected Move: {move} | Time: {elapsed:.3f}s | Root Entropy: {entropy:.4f}")
        print(game_state)
        print("-" * 40)
        
    print(f"Game Over! Winner is Player {game_state.check_winner()}")

--- Starting AI vs AI Test Match ---
. . . .
 . . . .
  . . . .
   . . . .
----------------------------------------
UCT (Player 1) is thinking...
[UCT] Selected Move: (np.int64(0), np.int64(3)) | Time: 0.155s | Root Entropy: 2.7273
. . . X
 . . . .
  . . . .
   . . . .
----------------------------------------
Thompson Sampling (Player 2) is thinking...
[Thompson] Selected Move: (np.int64(2), np.int64(2)) | Time: 0.390s | Root Entropy: 2.4466
. . . X
 . . . .
  . . O .
   . . . .
----------------------------------------
UCT (Player 1) is thinking...
[UCT] Selected Move: (np.int64(2), np.int64(1)) | Time: 0.168s | Root Entropy: 2.5457
. . . X
 . . . .
  . X O .
   . . . .
----------------------------------------
Thompson Sampling (Player 2) is thinking...
[Thompson] Selected Move: (np.int64(1), np.int64(0)) | Time: 0.150s | Root Entropy: 2.4668
. . . X
 O . . .
  . X O .
   . . . .
----------------------------------------
UCT (Player 1) is thinking...
[UCT] Selected Move: (np.int64(1), n

# Automated Tournament Script

In [11]:
import csv
import os
import time
import numpy as np

def clean_move(move):
    """Converts numpy int64 types to standard Python ints for clean logging."""
    return (int(move[0]), int(move[1]))

def run_tournament(game_class, board_size, num_games, iterations, csv_filename="tournament_results.csv"):
    """
    Runs an automated tournament between UCT and Thompson Sampling.
    Alternates which agent goes first to ensure fairness.
    """
    # Define file headers
    file_exists = os.path.isfile(csv_filename)
    
    with open(csv_filename, mode='a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow([
                "Game_Type", "Board_Size", "Game_ID", "Player_1_Strategy", "Player_2_Strategy", 
                "Winner", "Total_Moves", "Avg_UCT_Entropy", "Avg_TS_Entropy", "Avg_UCT_Time", "Avg_TS_Time"
            ])

        for game_id in range(1, num_games + 1):
            # Alternate opening player to eliminate first-mover advantage
            # Game 1: P1=UCT, P2=TS | Game 2: P1=TS, P2=UCT
            if game_id % 2 != 0:
                p1_strategy, p2_strategy = "UCT", "Thompson"
            else:
                p1_strategy, p2_strategy = "Thompson", "UCT"
                
            print(f"\nStarting Game {game_id}/{num_games}: Player 1 ({p1_strategy}) vs Player 2 ({p2_strategy})")
            
            state = game_class(board_size=board_size)
            
            # Metric accumulators for this specific game
            uct_entropies, ts_entropies = [], []
            uct_times, ts_times = [], []
            move_count = 0
            
            while not state.is_terminal():
                current_player_strategy = p1_strategy if state.turn == 1 else p2_strategy
                start_time = time.time()
                
                # Execute search based on current active strategy
                if current_player_strategy == "UCT":
                    move, root = run_mcts(state, iterations=iterations, selection_strategy=select_child_uct, exploration_constant=1.414)
                    elapsed = time.time() - start_time
                    uct_times.append(elapsed)
                    uct_entropies.append(calculate_root_entropy(root))
                else:
                    move, root = run_mcts(state, iterations=iterations, selection_strategy=select_child_thompson, prior_alpha=1.0, prior_beta=1.0)
                    elapsed = time.time() - start_time
                    ts_times.append(elapsed)
                    ts_entropies.append(calculate_root_entropy(root))
                
                # Clean and apply the move
                move = clean_move(move)
                state = state.make_move(move)
                move_count += 1
                
            winner_id = state.check_winner()
            # Map numerical winner back to the strategy name
            game_winner = p1_strategy if winner_id == 1 else (p2_strategy if winner_id == 2 else "Draw")
            
            # Calculate averages for the game
            avg_uct_entropy = np.mean(uct_entropies) if uct_entropies else 0
            avg_ts_entropy = np.mean(ts_entropies) if ts_entropies else 0
            avg_uct_time = np.mean(uct_times) if uct_times else 0
            avg_ts_time = np.mean(ts_times) if ts_times else 0
            
            print(f"Game {game_id} Finished! Winner: {game_winner} | Total Moves: {move_count}")
            
            # Save data row instantly (prevents data loss if program is interrupted)
            writer.writerow([
                game_class.__name__, board_size, game_id, p1_strategy, p2_strategy,
                game_winner, move_count, avg_uct_entropy, avg_ts_entropy, avg_uct_time, avg_ts_time
            ])
            f.flush() # Force write to disk

    print(f"\nTournament complete! Results successfully saved to '{csv_filename}'.")

In [12]:
if __name__ == "__main__":
    # 1. Run a mini Hex tournament (Board size 4x4, 200 iterations per move)
    print("Launching Hex Mini-Tournament...")
    run_tournament(HexState, board_size=4, num_games=4, iterations=200, csv_filename="hex_test_results.csv")
    
    # 2. Run a mini Gomoku tournament (Board size 6x6, 200 iterations per move)
    print("\nLaunching Gomoku Mini-Tournament...")
    run_tournament(GomokuState, board_size=6, num_games=4, iterations=200, csv_filename="gomoku_test_results.csv")

Launching Hex Mini-Tournament...

Starting Game 1/4: Player 1 (UCT) vs Player 2 (Thompson)
Game 1 Finished! Winner: UCT | Total Moves: 7

Starting Game 2/4: Player 1 (Thompson) vs Player 2 (UCT)
Game 2 Finished! Winner: UCT | Total Moves: 8

Starting Game 3/4: Player 1 (UCT) vs Player 2 (Thompson)
Game 3 Finished! Winner: UCT | Total Moves: 7

Starting Game 4/4: Player 1 (Thompson) vs Player 2 (UCT)
Game 4 Finished! Winner: UCT | Total Moves: 8

Tournament complete! Results successfully saved to 'hex_test_results.csv'.

Launching Gomoku Mini-Tournament...

Starting Game 1/4: Player 1 (UCT) vs Player 2 (Thompson)
Game 1 Finished! Winner: Thompson | Total Moves: 24

Starting Game 2/4: Player 1 (Thompson) vs Player 2 (UCT)
Game 2 Finished! Winner: Draw | Total Moves: 36

Starting Game 3/4: Player 1 (UCT) vs Player 2 (Thompson)
Game 3 Finished! Winner: Draw | Total Moves: 36

Starting Game 4/4: Player 1 (Thompson) vs Player 2 (UCT)
Game 4 Finished! Winner: Thompson | Total Moves: 19

Tourn

# Hyperparameter Tuning

In [13]:
import csv
import os
import numpy as np

def run_tuning_session(game_class, board_size, iterations, games_per_config=10, csv_filename="hyperparameter_tuning.csv"):
    """
    Evaluates different hyperparameters for UCT and Thompson Sampling 
    against a fixed baseline agent.
    """
    file_exists = os.path.isfile(csv_filename)
    
    with open(csv_filename, mode='a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["Game_Type", "Algorithm_Under_Test", "Parameter_Value", "Win_Rate", "Avg_Entropy", "Avg_Time"])

        # Baseline Agent: Standard UCT with standard C = 1.414
        baseline_strategy = select_child_uct
        baseline_kwargs = {"exploration_constant": 1.414}

        # -------------------------------------------------------------
        # TEST 1: Tuning UCT Exploration Constant (C)
        # -------------------------------------------------------------
        c_values = [0.5, 1.0, 1.414, 2.0, 2.5]
        print("--- Starting UCT Hyperparameter Tuning ---")
        
        for c in c_values:
            print(f"Testing UCT with C = {c}...")
            wins = 0
            entropies = []
            times = []
            
            for game_id in range(games_per_config):
                state = game_class(board_size=board_size)
                # Alternate turns: even games the test agent goes first, odd games baseline goes first
                test_agent_player = 1 if game_id % 2 == 0 else 2
                
                while not state.is_terminal():
                    import time
                    start_time = time.time()
                    
                    if state.turn == test_agent_player:
                        # Run MCTS with the target C value we are testing
                        move, root = run_mcts(state, iterations=iterations, 
                                              selection_strategy=select_child_uct, exploration_constant=c)
                        entropies.append(calculate_root_entropy(root))
                        times.append(time.time() - start_time)
                    else:
                        # Run MCTS with the baseline configuration
                        move, root = run_mcts(state, iterations=iterations, 
                                              selection_strategy=baseline_strategy, **baseline_kwargs)
                    
                    state = state.make_move(clean_move(move))
                    
                if state.check_winner() == test_agent_player:
                    wins += 1
            
            win_rate = wins / games_per_config
            writer.writerow([game_class.__name__, "UCT", f"C={c}", win_rate, np.mean(entropies), np.mean(times)])
            f.flush()

        # -------------------------------------------------------------
        # TEST 2: Tuning Thompson Sampling Priors (alpha, beta)
        # -------------------------------------------------------------
        priors = [
            ("Uniform(1,1)", 1.0, 1.0),
            ("Jeffreys(0.5,0.5)", 0.5, 0.5),
            ("Optimistic(2,1)", 2.0, 1.0),
            ("Balanced(2,2)", 2.0, 2.0)
        ]
        print("\n--- Starting Thompson Sampling Hyperparameter Tuning ---")
        
        for name, alpha, beta in priors:
            print(f"Testing Thompson Sampling with Prior = {name}...")
            wins = 0
            entropies = []
            times = []
            
            for game_id in range(games_per_config):
                state = game_class(board_size=board_size)
                test_agent_player = 1 if game_id % 2 == 0 else 2
                
                while not state.is_terminal():
                    import time
                    start_time = time.time()
                    
                    if state.turn == test_agent_player:
                        # Run MCTS with the target Beta priors we are testing
                        move, root = run_mcts(state, iterations=iterations, 
                                              selection_strategy=select_child_thompson, prior_alpha=alpha, prior_beta=beta)
                        entropies.append(calculate_root_entropy(root))
                        times.append(time.time() - start_time)
                    else:
                        # Run MCTS with the baseline configuration
                        move, root = run_mcts(state, iterations=iterations, 
                                              selection_strategy=baseline_strategy, **baseline_kwargs)
                    
                    state = state.make_move(clean_move(move))
                    
                if state.check_winner() == test_agent_player:
                    wins += 1
            
            win_rate = wins / games_per_config
            writer.writerow([game_class.__name__, "ThompsonSampling", name, win_rate, np.mean(entropies), np.mean(times)])
            f.flush()

    print(f"\nTuning session complete! Results saved to '{csv_filename}'.")

In [14]:
if __name__ == "__main__":
    # Run tuning on the small Hex board configuration
    run_tuning_session(HexState, board_size=4, iterations=200, games_per_config=10, csv_filename="hex_tuning_results.csv")

--- Starting UCT Hyperparameter Tuning ---
Testing UCT with C = 0.5...
Testing UCT with C = 1.0...
Testing UCT with C = 1.414...
Testing UCT with C = 2.0...
Testing UCT with C = 2.5...

--- Starting Thompson Sampling Hyperparameter Tuning ---
Testing Thompson Sampling with Prior = Uniform(1,1)...
Testing Thompson Sampling with Prior = Jeffreys(0.5,0.5)...
Testing Thompson Sampling with Prior = Optimistic(2,1)...
Testing Thompson Sampling with Prior = Balanced(2,2)...

Tuning session complete! Results saved to 'hex_tuning_results.csv'.


# Medium-Scale Tournament

In [15]:
import csv
import os
import time
import numpy as np

def clean_move(move):
    """Converts numpy int64 types to standard Python ints for clean logging."""
    return (int(move[0]), int(move[1]))

def run_medium_scale_tournament(game_class, board_size=6, num_games=20, iterations=1000, csv_filename="medium_scale_results.csv"):
    """
    Runs a medium-scale tournament between the OPTIMIZED UCT and Thompson Sampling agents.
    Alternates player roles each game to eliminate first-mover advantage.
    """
    file_exists = os.path.isfile(csv_filename)
    
    print(f"==================================================")
    print(f"LAUNCHING MEDIUM-SCALE TOURNAMENT")
    print(f"Game: {game_class.__name__} | Board Size: {board_size}x{board_size}")
    print(f"Total Games: {num_games} | Iteration Budget: {iterations}")
    print(f"Optimized UCT: C = 1.0")
    print(f"Optimized Thompson Sampling: Prior = Optimistic (2,1)")
    print(f"==================================================")

    with open(csv_filename, mode='a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow([
                "Game_Type", "Board_Size", "Game_ID", "Player_1_Strategy", "Player_2_Strategy", 
                "Winner", "Total_Moves", "Avg_UCT_Entropy", "Avg_TS_Entropy", "Avg_UCT_Time", "Avg_TS_Time"
            ])

        for game_id in range(1, num_games + 1):
            # Alternate opening turns for absolute fairness
            if game_id % 2 != 0:
                p1_strategy, p2_strategy = "UCT", "Thompson"
            else:
                p1_strategy, p2_strategy = "Thompson", "UCT"
                
            print(f"Game {game_id}/{num_games}: P1 ({p1_strategy}) vs P2 ({p2_strategy})...", end="", flush=True)
            
            state = game_class(board_size=board_size)
            
            # Metrics arrays
            uct_entropies, ts_entropies = [], []
            uct_times, ts_times = [], []
            move_count = 0
            
            game_start = time.time()
            
            while not state.is_terminal():
                current_player_strategy = p1_strategy if state.turn == 1 else p2_strategy
                step_start = time.time()
                
                if current_player_strategy == "UCT":
                    # Using optimized C = 1.0
                    move, root = run_mcts(state, iterations=iterations, 
                                          selection_strategy=select_child_uct, exploration_constant=1.0)
                    uct_times.append(time.time() - step_start)
                    uct_entropies.append(calculate_root_entropy(root))
                else:
                    # Using optimized Optimistic Prior (2,1)
                    move, root = run_mcts(state, iterations=iterations, 
                                          selection_strategy=select_child_thompson, prior_alpha=2.0, prior_beta=1.0)
                    ts_times.append(time.time() - step_start)
                    ts_entropies.append(calculate_root_entropy(root))
                
                state = state.make_move(clean_move(move))
                move_count += 1
                
            winner_id = state.check_winner()
            game_winner = p1_strategy if winner_id == 1 else (p2_strategy if winner_id == 2 else "Draw")
            elapsed_game_time = time.time() - game_start
            
            print(f" Finished! Winner: {game_winner} ({move_count} moves, {elapsed_game_time:.1f}s)")
            
            # Log metrics to CSV immediately
            writer.writerow([
                game_class.__name__, board_size, game_id, p1_strategy, p2_strategy,
                game_winner, move_count, 
                np.mean(uct_entropies) if uct_entropies else 0, 
                np.mean(ts_entropies) if ts_entropies else 0, 
                np.mean(uct_times) if uct_times else 0, 
                np.mean(ts_times) if ts_times else 0
            ])
            f.flush()

    print(f"\nTournament complete! Results saved to '{csv_filename}'.")

if __name__ == "__main__":
    # Run the tournament on a 6x6 Hex board
    run_medium_scale_tournament(HexState, board_size=6, num_games=20, iterations=1000, csv_filename="medium_scale_results.csv")

LAUNCHING MEDIUM-SCALE TOURNAMENT
Game: HexState | Board Size: 6x6
Total Games: 20 | Iteration Budget: 1000
Optimized UCT: C = 1.0
Optimized Thompson Sampling: Prior = Optimistic (2,1)
Game 1/20: P1 (UCT) vs P2 (Thompson)... Finished! Winner: UCT (15 moves, 14.1s)
Game 2/20: P1 (Thompson) vs P2 (UCT)... Finished! Winner: Thompson (21 moves, 18.8s)
Game 3/20: P1 (UCT) vs P2 (Thompson)... Finished! Winner: Thompson (14 moves, 14.8s)
Game 4/20: P1 (Thompson) vs P2 (UCT)... Finished! Winner: Thompson (15 moves, 13.1s)
Game 5/20: P1 (UCT) vs P2 (Thompson)... Finished! Winner: UCT (15 moves, 15.3s)
Game 6/20: P1 (Thompson) vs P2 (UCT)... Finished! Winner: Thompson (15 moves, 12.6s)
Game 7/20: P1 (UCT) vs P2 (Thompson)... Finished! Winner: UCT (15 moves, 16.8s)
Game 8/20: P1 (Thompson) vs P2 (UCT)... Finished! Winner: UCT (16 moves, 17.0s)
Game 9/20: P1 (UCT) vs P2 (Thompson)... Finished! Winner: Thompson (12 moves, 9.8s)
Game 10/20: P1 (Thompson) vs P2 (UCT)... Finished! Winner: UCT (16 move

# Graphs and Metrics for Medium Scale Tournament

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configure aesthetic style for academic reporting
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.titlesize": 16
})

def analyze_and_plot_results(csv_filename="medium_scale_results.csv"):
    # 1. Load the dataset
    if not os.path.exists(csv_filename):
        print(f"Error: Could not find '{csv_filename}'. Please ensure it is in the same directory.")
        return
        
    df = pd.read_csv(csv_filename)
    
    # 2. Print Descriptive Statistical Summary
    print("=" * 60)
    print("             MCTS TOURNAMENT DATA STATISTICAL REPORT          ")
    print("=" * 60)
    
    total_games = len(df)
    uct_wins = len(df[df['Winner'] == 'UCT'])
    ts_wins = len(df[df['Winner'] == 'Thompson'])
    draws = len(df[df['Winner'] == 'Draw'])
    
    print(f"Total Games Played: {total_games}")
    print(f"UCT Total Wins:     {uct_wins} ({uct_wins/total_games*100:.1f}%)")
    print(f"Thompson Total Wins: {ts_wins} ({ts_wins/total_games*100:.1f}%)")
    print(f"Draws:               {draws} ({draws/total_games*100:.1f}%)")
    print("-" * 60)
    
    print("\n--- Mean Performance Metrics ---")
    print(f"Average Total Moves per Game: {df['Total_Moves'].mean():.2f}")
    print(f"Average UCT Root Entropy:    {df['Avg_UCT_Entropy'].mean():.4f}")
    print(f"Average TS Root Entropy:     {df['Avg_TS_Entropy'].mean():.4f}")
    print(f"Average UCT Move Latency:    {df['Avg_UCT_Time'].mean():.4f} seconds")
    print(f"Average TS Move Latency:     {df['Avg_TS_Time'].mean():.4f} seconds")
    print("=" * 60)

    # 3. Plot 1: Win Distribution Bar Chart
    plt.figure(figsize=(6, 5))
    win_counts = df['Winner'].value_counts()
    sns.barplot(x=win_counts.index, y=win_counts.values, hue=win_counts.index, legend=False, palette="muted")
    plt.title("Win-Loss Distribution")
    plt.xlabel("Strategy")
    plt.ylabel("Total Wins")
    plt.tight_layout()
    plt.savefig("mcts_win_distribution.png", dpi=300)
    plt.close()
    
    # 4. Plot 2: Decision Latency Comparison (Boxplot)
    plt.figure(figsize=(7, 5))
    time_data = pd.melt(df, id_vars=['Game_ID'], value_vars=['Avg_UCT_Time', 'Avg_TS_Time'],
                        var_name='Strategy', value_name='Time')
    time_data['Strategy'] = time_data['Strategy'].map({'Avg_UCT_Time': 'UCT (C=1.0)', 'Avg_TS_Time': 'Thompson (2,1)'})
    
    sns.boxplot(data=time_data, x='Strategy', y='Time', hue='Strategy', legend=False, palette="pastel", width=0.5)
    sns.stripplot(data=time_data, x='Strategy', y='Time', color="black", alpha=0.5, jitter=0.1)
    plt.title("Computational Efficiency: Move Decision Latency")
    plt.xlabel("MCTS Variant Configuration")
    plt.ylabel("Wall-Clock Execution Time (Seconds / Decision)")
    plt.tight_layout()
    plt.savefig("mcts_efficiency_comparison.png", dpi=300)
    plt.close()

    # 5. Plot 3: Shannon Entropy Lifespan Variance (Line Plot over Games)
    plt.figure(figsize=(10, 5))
    plt.plot(df['Game_ID'], df['Avg_UCT_Entropy'], marker='o', linestyle='-', label='UCT (C=1.0)', linewidth=2)
    plt.plot(df['Game_ID'], df['Avg_TS_Entropy'], marker='s', linestyle='--', label='Thompson (2,1)', linewidth=2)
    plt.title("Search Tree Branching Diversity: Root Node Shannon Entropy")
    plt.xlabel("Game Session ID")
    plt.ylabel("Average Shannon Entropy (Bits)")
    plt.xticks(df['Game_ID'])
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig("mcts_entropy_variance.png", dpi=300)
    plt.close()
    
    print("\nVisualizations successfully saved to directory:")
    print(" -> 'mcts_win_distribution.png'")
    print(" -> 'mcts_efficiency_comparison.png'")
    print(" -> 'mcts_entropy_variance.png'")

if __name__ == "__main__":
    # Run analysis
    analyze_and_plot_results()

             MCTS TOURNAMENT DATA STATISTICAL REPORT          
Total Games Played: 20
UCT Total Wins:     10 (50.0%)
Thompson Total Wins: 10 (50.0%)
Draws:               0 (0.0%)
------------------------------------------------------------

--- Mean Performance Metrics ---
Average Total Moves per Game: 16.80
Average UCT Root Entropy:    3.1913
Average TS Root Entropy:     2.8811
Average UCT Move Latency:    0.9769 seconds
Average TS Move Latency:     0.9185 seconds

Visualizations successfully saved to directory:
 -> 'mcts_win_distribution.png'
 -> 'mcts_efficiency_comparison.png'
 -> 'mcts_entropy_variance.png'


# Stats for Hex Large Scale Tournament

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chisquare

# Set clean, academic plot configurations
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 16
})

def analyze_tournament_data(csv_path):
    # 1. Load the dataset
    df = pd.read_csv(csv_path)
    total_games = len(df)
    
    print("=" * 70)
    print(f"         MCTS MASTER DATA ANALYSIS REPORT ({total_games} GAMES OVERALL)")
    print("=" * 70)
    
    # =====================================================================
    # ANALYSIS 1: WIN RATES AND FIRST-PLAYER BIAS
    # =====================================================================
    uct_wins = len(df[df['Winner'] == 'UCT'])
    ts_wins = len(df[df['Winner'] == 'Thompson'])
    draws = len(df[df['Winner'] == 'Draw'])
    
    # Calculate First-Player (P1) Wins vs Second-Player (P2) Wins
    p1_wins = len(df[df['Winner'] == df['Player_1_Strategy']])
    p2_wins = len(df[df['Winner'] == df['Player_2_Strategy']])
    
    print("\n--- 1. HEAD-TO-HEAD WIN METRICS ---")
    print(f"UCT Wins:              {uct_wins} ({uct_wins/total_games*100:.1f}%)")
    print(f"Thompson Sampling Wins: {ts_wins} ({ts_wins/total_games*100:.1f}%)")
    if draws > 0:
        print(f"Draws:                 {draws} ({draws/total_games*100:.1f}%)")
        
    print(f"\nFirst Player (P1) Wins:  {p1_wins} ({p1_wins/total_games*100:.1f}%)")
    print(f"Second Player (P2) Wins: {p2_wins} ({p2_wins/total_games*100:.1f}%)")
    
    # Statistical validation (Chi-Square Goodness of Fit test against a 50/50 balance)
    chi_stat, p_val = chisquare([p1_wins, p2_wins], f_exp=[total_games/2, total_games/2])
    print(f"First-Mover Bias Check (p-value): {p_val:.4f}")
    if p_val > 0.05:
        print(" -> Verdict: No statistically significant first-mover bias. Fairness structure is valid.")
    else:
        print(" -> Verdict: Statistically significant first-mover bias detected.")

    # =====================================================================
    # ANALYSIS 2: ALGORITHMIC METRICS (ENTROPY & LATENCY)
    # =====================================================================
    print("\n--- 2. SEARCH BEHAVIOR METRICS ---")
    print(f"UCT Mean Root Entropy:              {df['Avg_UCT_Entropy'].mean():.4f}")
    print(f"Thompson Sampling Mean Root Entropy: {df['Avg_TS_Entropy'].mean():.4f}")
    print(f"UCT Mean Time per Turn:              {df['Avg_UCT_Time'].mean():.2f} seconds")
    print(f"Thompson Sampling Mean Time per Turn: {df['Avg_TS_Time'].mean():.2f} seconds")
    print(f"Average Moves per Game:              {df['Total_Moves'].mean():.1f} steps")
    print("=" * 70)

    # =====================================================================
    # PLOTTING CODES
    # =====================================================================
    # Plot 1: Win Distribution Bar Chart
    plt.figure(figsize=(6, 5))
    strategies = ['UCT', 'Thompson Sampling']
    win_counts = [uct_wins, ts_wins]
    colors = ['#1f77b4', '#ff7f0e']
    
    bars = plt.bar(strategies, win_counts, color=colors, edgecolor='black', width=0.6)
    plt.ylabel('Total Tournament Wins')
    plt.title('Head-to-Head Win Distribution\n(11x11 Hex, 2500 Iterations)')
    
    # Add numbers on top of bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + (total_games*0.01),
                 f'{int(height)}\n({height/total_games*100:.1f}%)',
                 ha='center', va='bottom', fontweight='bold')
                 
    plt.tight_layout()
    plt.savefig('hex_win_distribution.png', dpi=300)
    plt.close()

    # Plot 2: Root Entropy Distributions (Search Depth Context)
    plt.figure(figsize=(7, 5))
    sns.kdeplot(data=df['Avg_UCT_Entropy'], fill=True, color='#1f77b4', label='UCT Entropy', linewidth=2)
    sns.kdeplot(data=df['Avg_TS_Entropy'], fill=True, color='#ff7f0e', label='Thompson Sampling Entropy', linewidth=2)
    plt.xlabel('Mean Shannon Entropy ($H$)')
    plt.ylabel('Density Probability Profile')
    plt.title('Algorithmic Decision Certainty Profiles (Root Node Entropy)')
    plt.legend(loc='upper left')
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig('hex_entropy_comparison.png', dpi=300)
    plt.close()

    # Plot 3: Game Length vs. Computational Footprint Scatter Plot
    plt.figure(figsize=(7, 5))
    plt.scatter(df['Total_Moves'], df['Avg_UCT_Time'], color='#1f77b4', alpha=0.6, edgecolors='none', label='UCT')
    plt.scatter(df['Total_Moves'], df['Avg_TS_Time'], color='#ff7f0e', alpha=0.6, edgecolors='none', label='Thompson')
    
    # Trendlines
    for time_col, col, lbl in [('Avg_UCT_Time', '#1f77b4', 'UCT Trend'), ('Avg_TS_Time', '#ff7f0e', 'TS Trend')]:
        z = np.polyfit(df['Total_Moves'], df[time_col], 1)
        p = np.poly1d(z)
        xp = np.linspace(df['Total_Moves'].min(), df['Total_Moves'].max(), 100)
        plt.plot(xp, p(xp), color=col, linestyle='--', linewidth=1.5, label=lbl)

    plt.xlabel('Total Game Duration (Total Moves)')
    plt.ylabel('Average Turn Execution Latency (Seconds)')
    plt.title('Computational Complexity Scaling Framework')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.savefig('hex_latency_complexity.png', dpi=300)
    plt.close()
    
    print("\n[SUCCESS] Generated 3 data visualizations:")
    print(" -> 'hex_win_distribution.png'")
    print(" -> 'hex_entropy_comparison.png'")
    print(" -> 'hex_latency_complexity.png'")

if __name__ == "__main__":
    # Point this file path to your combined master csv location
    analyze_tournament_data("csf_hex_11x11_FINAL.csv")

         MCTS MASTER DATA ANALYSIS REPORT (500 GAMES OVERALL)

--- 1. HEAD-TO-HEAD WIN METRICS ---
UCT Wins:              179 (35.8%)
Thompson Sampling Wins: 321 (64.2%)

First Player (P1) Wins:  253 (50.6%)
Second Player (P2) Wins: 247 (49.4%)
First-Mover Bias Check (p-value): 0.7884
 -> Verdict: No statistically significant first-mover bias. Fairness structure is valid.

--- 2. SEARCH BEHAVIOR METRICS ---
UCT Mean Root Entropy:              4.5098
Thompson Sampling Mean Root Entropy: 4.3300
UCT Mean Time per Turn:              36.01 seconds
Thompson Sampling Mean Time per Turn: 35.28 seconds
Average Moves per Game:              48.3 steps

[SUCCESS] Generated 3 data visualizations:
 -> 'hex_win_distribution.png'
 -> 'hex_entropy_comparison.png'
 -> 'hex_latency_complexity.png'


# Stats for Large Scale Gomoku Tournament

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chisquare

# Set academic visualization standards
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 15
})

def analyze_gomoku_results(csv_path):
    # Load dataset
    df = pd.read_csv(csv_path)
    total_games = len(df)
    
    print("=" * 70)
    print(f"       MCTS GOMOKU 15x15 MASTER ANALYSIS REPORT ({total_games} GAMES)")
    print("=" * 70)
    
    # Win percentages
    uct_wins = len(df[df['Winner'] == 'UCT'])
    ts_wins = len(df[df['Winner'] == 'Thompson'])
    draw_wins = len(df[df['Winner'] == 'Draw'])
    
    # Seat orientations
    p1_wins = len(df[df['Winner'] == df['Player_1_Strategy']])
    p2_wins = len(df[df['Winner'] == df['Player_2_Strategy']])
    
    print("\n--- 1. HEAD-TO-HEAD WIN PROFILE ---")
    print(f"UCT Wins:              {uct_wins:3d} ({uct_wins/total_games*100:.2f}%)")
    print(f"Thompson Sampling Wins: {ts_wins:3d} ({ts_wins/total_games*100:.2f}%)")
    print(f"Draws:                 {draw_wins:3d} ({draw_wins/total_games*100:.2f}%)")
    
    print("\n--- 2. GEOMETRIC FIRST-MOVER BIAS ---")
    print(f"Player 1 (Goes First) Wins:  {p1_wins:3d} ({p1_wins/total_games*100:.2f}%)")
    print(f"Player 2 (Goes Second) Wins: {p2_wins:3d} ({p2_wins/total_games*100:.2f}%)")
    
    # Chi-square validation for first-mover advantage
    chi_stat, p_val = chisquare([p1_wins, p2_wins], f_exp=[total_games/2, total_games/2])
    print(f"Pearson Chi-Square p-value:  {p_val:.5f}")
    if p_val > 0.05:
        print(" -> VERDICT: No statistically significant first-mover bias. Structural alternation successfully neutralized board geometry.")
    else:
        print(" -> WARNING: Significant first-mover advantage detected.")
        
    print("\n--- 3. INFORMATION SEARCH & SEARCH COMPACTNESS ---")
    print(f"UCT Mean Root Shannon Entropy:             {df['Avg_UCT_Entropy'].mean():.5f}")
    print(f"Thompson Sampling Mean Root Shannon Entropy: {df['Avg_TS_Entropy'].mean():.5f}")
    print(f"UCT Mean Execution Time per Move:          {df['Avg_UCT_Time'].mean():.3f} seconds")
    print(f"Thompson Sampling Mean Time per Move:       {df['Avg_TS_Time'].mean():.3f} seconds")
    print(f"Average Total Moves per Game:              {df['Total_Moves'].mean():.2f} turns")
    print("=" * 70)

    # Plot 1: Win distribution bar chart
    plt.figure(figsize=(6, 5))
    categories = ['UCT', 'Thompson Sampling']
    wins = [uct_wins, ts_wins]
    colors = ['#4682B4', '#E67E22']
    bars = plt.bar(categories, wins, color=colors, edgecolor='#2C3E50', width=0.55)
    plt.ylabel('Total Wins')
    plt.title('Head-to-Head Win Balance\n(15x15 Gomoku Tournament)')
    for bar in bars:
        h = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., h + (total_games*0.015),
                 f'{int(h)}\n({h/total_games*100:.1f}%)',
                 ha='center', va='bottom', fontweight='bold')
    plt.tight_layout()
    plt.savefig('gomoku_win_distribution.png', dpi=300)
    plt.close()

    # Plot 2: Shannon Entropy PDF Distributions
    plt.figure(figsize=(7, 4.5))
    sns.kdeplot(data=df['Avg_UCT_Entropy'], fill=True, color='#4682B4', label='UCT Entropy', linewidth=2.5)
    sns.kdeplot(data=df['Avg_TS_Entropy'], fill=True, color='#E67E22', label='Thompson Sampling Entropy', linewidth=2.5)
    plt.xlabel('Shannon Information Entropy (H)')
    plt.ylabel('Probability Density Estimate')
    plt.title('Algorithmic Information Variance (Root Node Shannon Entropy)')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.savefig('gomoku_entropy_distribution.png', dpi=300)
    plt.close()

    # Plot 3: Computational Footprint Scaling Frame
    plt.figure(figsize=(7, 4.5))
    plt.scatter(df['Total_Moves'], df['Avg_UCT_Time'], color='#4682B4', alpha=0.5, label='UCT Time')
    plt.scatter(df['Total_Moves'], df['Avg_TS_Time'], color='#E67E22', alpha=0.5, label='Thompson Sampling Time')
    for time_col, col, lbl in [('Avg_UCT_Time', '#4682B4', 'UCT Fit'), ('Avg_TS_Time', '#E67E22', 'TS Fit')]:
        z = np.polyfit(df['Total_Moves'], df[time_col], 1)
        p = np.poly1d(z)
        xp = np.linspace(df['Total_Moves'].min(), df['Total_Moves'].max(), 100)
        plt.plot(xp, p(xp), color=col, linestyle='--', linewidth=1.8, label=lbl)
    plt.xlabel('Total Game Length (Moves)')
    plt.ylabel('Mean Turn Execution Latency (Seconds)')
    plt.title('Computational Footprint Latency Scaling')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.savefig('gomoku_latency_scaling.png', dpi=300)
    plt.close()
    print("\n[VISUALIZATION SUCCESS] Exported:")
    print(" -> 'gomoku_win_distribution.png'")
    print(" -> 'gomoku_entropy_distribution.png'")
    print(" -> 'gomoku_latency_scaling.png'")

if __name__ == "__main__":
    analyze_gomoku_results("csf_gomoku_15x15_FINAL.csv")

       MCTS GOMOKU 15x15 MASTER ANALYSIS REPORT (500 GAMES)

--- 1. HEAD-TO-HEAD WIN PROFILE ---
UCT Wins:              192 (38.40%)
Thompson Sampling Wins: 308 (61.60%)
Draws:                   0 (0.00%)

--- 2. GEOMETRIC FIRST-MOVER BIAS ---
Player 1 (Goes First) Wins:  244 (48.80%)
Player 2 (Goes Second) Wins: 256 (51.20%)
Pearson Chi-Square p-value:  0.59151
 -> VERDICT: No statistically significant first-mover bias. Structural alternation successfully neutralized board geometry.

--- 3. INFORMATION SEARCH & SEARCH COMPACTNESS ---
UCT Mean Root Shannon Entropy:             5.28807
Thompson Sampling Mean Root Shannon Entropy: 5.07158
UCT Mean Execution Time per Move:          18.094 seconds
Thompson Sampling Mean Time per Move:       17.021 seconds
Average Total Moves per Game:              30.57 turns

[VISUALIZATION SUCCESS] Exported:
 -> 'gomoku_win_distribution.png'
 -> 'gomoku_entropy_distribution.png'
 -> 'gomoku_latency_scaling.png'


# Short Budget Full Tournaments (250 iterations)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chisquare

# Set clean, academic plot configurations
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 15
})

def analyze_game_data(csv_path, game_title, iterations):
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"[ERROR] Could not find {csv_path}. Please check the filename.")
        return

    total_games = len(df)
    
    print("=" * 70)
    print(f"         {game_title.upper()} ANALYSIS REPORT ({total_games} GAMES | {iterations} ITERATIONS)")
    print("=" * 70)
    
    # 1. Win Rates
    uct_wins = len(df[df['Winner'] == 'UCT'])
    ts_wins = len(df[df['Winner'] == 'Thompson'])
    draws = len(df[df['Winner'] == 'Draw'])
    
    p1_wins = len(df[df['Winner'] == df['Player_1_Strategy']])
    p2_wins = len(df[df['Winner'] == df['Player_2_Strategy']])
    
    print("\n--- 1. HEAD-TO-HEAD WIN METRICS ---")
    print(f"UCT Wins:               {uct_wins:3d} ({uct_wins/total_games*100:.1f}%)")
    print(f"Thompson Sampling Wins: {ts_wins:3d} ({ts_wins/total_games*100:.1f}%)")
    if draws > 0:
        print(f"Draws:                  {draws:3d} ({draws/total_games*100:.1f}%)")
        
    print(f"\nFirst Player (P1) Wins:  {p1_wins:3d} ({p1_wins/total_games*100:.1f}%)")
    print(f"Second Player (P2) Wins: {p2_wins:3d} ({p2_wins/total_games*100:.1f}%)")
    
    chi_stat, p_val = chisquare([p1_wins, p2_wins], f_exp=[total_games/2, total_games/2])
    print(f"First-Mover Bias (p-value): {p_val:.5f}")
    if p_val > 0.05:
        print(" -> Verdict: No statistically significant bias.")
    else:
        print(" -> Verdict: Significant bias detected.")

    # 2. Search Behavior
    print("\n--- 2. SEARCH BEHAVIOR METRICS ---")
    print(f"UCT Mean Root Entropy:              {df['Avg_UCT_Entropy'].mean():.4f}")
    print(f"Thompson Sampling Mean Root Entropy: {df['Avg_TS_Entropy'].mean():.4f}")
    print(f"UCT Mean Time per Turn:              {df['Avg_UCT_Time'].mean():.2f} sec")
    print(f"Thompson Sampling Mean Time per Turn: {df['Avg_TS_Time'].mean():.2f} sec")
    print(f"Average Moves per Game:              {df['Total_Moves'].mean():.1f} steps")
    print("=" * 70 + "\n")

    # Plot 1: Win Distribution
    plt.figure(figsize=(6, 5))
    strategies = ['UCT', 'Thompson Sampling']
    win_counts = [uct_wins, ts_wins]
    colors = ['#1f77b4', '#ff7f0e']
    bars = plt.bar(strategies, win_counts, color=colors, edgecolor='black', width=0.6)
    plt.ylabel('Total Wins')
    plt.title(f'Head-to-Head Win Distribution\n({game_title}, {iterations} Iterations)')
    
    for bar in bars:
        h = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., h + (total_games*0.015),
                 f'{int(h)}\n({h/total_games*100:.1f}%)',
                 ha='center', va='bottom', fontweight='bold')
                 
    plt.tight_layout()
    plt.savefig(f'{game_title.lower()}_iter{iterations}_wins.png', dpi=300)
    plt.close()

    # Plot 2: Entropy
    plt.figure(figsize=(7, 5))
    sns.kdeplot(data=df['Avg_UCT_Entropy'], fill=True, color='#1f77b4', label='UCT Entropy', linewidth=2)
    sns.kdeplot(data=df['Avg_TS_Entropy'], fill=True, color='#ff7f0e', label='TS Entropy', linewidth=2)
    plt.xlabel('Mean Shannon Entropy (H)')
    plt.ylabel('Density Probability Profile')
    plt.title(f'Decision Certainty Profiles ({game_title})')
    plt.legend(loc='upper left')
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f'{game_title.lower()}_iter{iterations}_entropy.png', dpi=300)
    plt.close()

    # Plot 3: Latency
    plt.figure(figsize=(7, 5))
    plt.scatter(df['Total_Moves'], df['Avg_UCT_Time'], color='#1f77b4', alpha=0.5, edgecolors='none', label='UCT')
    plt.scatter(df['Total_Moves'], df['Avg_TS_Time'], color='#ff7f0e', alpha=0.5, edgecolors='none', label='Thompson')
    
    for time_col, col, lbl in [('Avg_UCT_Time', '#1f77b4', 'UCT Trend'), ('Avg_TS_Time', '#ff7f0e', 'TS Trend')]:
        z = np.polyfit(df['Total_Moves'], df[time_col], 1)







































        
        p = np.poly1d(z)
        xp = np.linspace(df['Total_Moves'].min(), df['Total_Moves'].max(), 100)
        plt.plot(xp, p(xp), color=col, linestyle='--', linewidth=1.5, label=lbl)

    plt.xlabel('Total Game Duration (Total Moves)')
    plt.ylabel('Average Turn Execution Latency (Seconds)')
    plt.title(f'Complexity Scaling ({game_title})')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.savefig(f'{game_title.lower()}_iter{iterations}_latency.png', dpi=300)
    plt.close()

if __name__ == "__main__":
    # Update these filenames if your low-budget CSVs are named differently
    analyze_game_data("csf_hex_11x11_FINAL.csv", "Hex", 250)
    analyze_game_data("csf_gomoku_15x15_FINAL.csv", "Gomoku", 250)

         HEX ANALYSIS REPORT (500 GAMES | 250 ITERATIONS)

--- 1. HEAD-TO-HEAD WIN METRICS ---
UCT Wins:               121 (24.2%)
Thompson Sampling Wins: 379 (75.8%)

First Player (P1) Wins:  245 (49.0%)
Second Player (P2) Wins: 255 (51.0%)
First-Mover Bias (p-value): 0.65472
 -> Verdict: No statistically significant bias.

--- 2. SEARCH BEHAVIOR METRICS ---
UCT Mean Root Entropy:              4.3089
Thompson Sampling Mean Root Entropy: 4.1492
UCT Mean Time per Turn:              2.86 sec
Thompson Sampling Mean Time per Turn: 2.83 sec
Average Moves per Game:              78.6 steps

         GOMOKU ANALYSIS REPORT (500 GAMES | 250 ITERATIONS)

--- 1. HEAD-TO-HEAD WIN METRICS ---
UCT Wins:               115 (23.0%)
Thompson Sampling Wins: 385 (77.0%)

First Player (P1) Wins:  227 (45.4%)
Second Player (P2) Wins: 273 (54.6%)
First-Mover Bias (p-value): 0.03967
 -> Verdict: Significant bias detected.

--- 2. SEARCH BEHAVIOR METRICS ---
UCT Mean Root Entropy:              5.1799
Thompson 

# High Budget Full Tournaments

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chisquare

# Set clean, academic plot configurations
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 15
})

def analyze_game_data(csv_path, game_title, iterations):
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"[ERROR] Could not find {csv_path}. Please check the filename.")
        return

    total_games = len(df)
    
    print("=" * 70)
    print(f"         {game_title.upper()} ANALYSIS REPORT ({total_games} GAMES | {iterations} ITERATIONS)")
    print("=" * 70)
    
    # 1. Win Rates
    uct_wins = len(df[df['Winner'] == 'UCT'])
    ts_wins = len(df[df['Winner'] == 'Thompson'])
    draws = len(df[df['Winner'] == 'Draw'])
    
    p1_wins = len(df[df['Winner'] == df['Player_1_Strategy']])
    p2_wins = len(df[df['Winner'] == df['Player_2_Strategy']])
    
    print("\n--- 1. HEAD-TO-HEAD WIN METRICS ---")
    print(f"UCT Wins:               {uct_wins:3d} ({uct_wins/total_games*100:.1f}%)")
    print(f"Thompson Sampling Wins: {ts_wins:3d} ({ts_wins/total_games*100:.1f}%)")
    if draws > 0:
        print(f"Draws:                  {draws:3d} ({draws/total_games*100:.1f}%)")
        
    print(f"\nFirst Player (P1) Wins:  {p1_wins:3d} ({p1_wins/total_games*100:.1f}%)")
    print(f"Second Player (P2) Wins: {p2_wins:3d} ({p2_wins/total_games*100:.1f}%)")
    
    chi_stat, p_val = chisquare([p1_wins, p2_wins], f_exp=[total_games/2, total_games/2])
    print(f"First-Mover Bias (p-value): {p_val:.5f}")
    if p_val > 0.05:
        print(" -> Verdict: No statistically significant bias.")
    else:
        print(" -> Verdict: Significant bias detected.")

    # 2. Search Behavior
    print("\n--- 2. SEARCH BEHAVIOR METRICS ---")
    print(f"UCT Mean Root Entropy:              {df['Avg_UCT_Entropy'].mean():.4f}")
    print(f"Thompson Sampling Mean Root Entropy: {df['Avg_TS_Entropy'].mean():.4f}")
    print(f"UCT Mean Time per Turn:              {df['Avg_UCT_Time'].mean():.2f} sec")
    print(f"Thompson Sampling Mean Time per Turn: {df['Avg_TS_Time'].mean():.2f} sec")
    print(f"Average Moves per Game:              {df['Total_Moves'].mean():.1f} steps")
    print("=" * 70 + "\n")

    # Plot 1: Win Distribution
    plt.figure(figsize=(6, 5))
    strategies = ['UCT', 'Thompson Sampling']
    win_counts = [uct_wins, ts_wins]
    colors = ['#1f77b4', '#ff7f0e']
    bars = plt.bar(strategies, win_counts, color=colors, edgecolor='black', width=0.6)
    plt.ylabel('Total Wins')
    plt.title(f'Head-to-Head Win Distribution\n({game_title}, {iterations} Iterations)')
    
    for bar in bars:
        h = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., h + (total_games*0.015),
                 f'{int(h)}\n({h/total_games*100:.1f}%)',
                 ha='center', va='bottom', fontweight='bold')
                 
    plt.tight_layout()
    plt.savefig(f'{game_title.lower()}_iter{iterations}_wins.png', dpi=300)
    plt.close()

    # Plot 2: Entropy
    plt.figure(figsize=(7, 5))
    sns.kdeplot(data=df['Avg_UCT_Entropy'], fill=True, color='#1f77b4', label='UCT Entropy', linewidth=2)
    sns.kdeplot(data=df['Avg_TS_Entropy'], fill=True, color='#ff7f0e', label='TS Entropy', linewidth=2)
    plt.xlabel('Mean Shannon Entropy (H)')
    plt.ylabel('Density Probability Profile')
    plt.title(f'Decision Certainty Profiles ({game_title})')
    plt.legend(loc='upper left')
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f'{game_title.lower()}_iter{iterations}_entropy.png', dpi=300)
    plt.close()

    # Plot 3: Latency
    plt.figure(figsize=(7, 5))
    plt.scatter(df['Total_Moves'], df['Avg_UCT_Time'], color='#1f77b4', alpha=0.5, edgecolors='none', label='UCT')
    plt.scatter(df['Total_Moves'], df['Avg_TS_Time'], color='#ff7f0e', alpha=0.5, edgecolors='none', label='Thompson')
    
    for time_col, col, lbl in [('Avg_UCT_Time', '#1f77b4', 'UCT Trend'), ('Avg_TS_Time', '#ff7f0e', 'TS Trend')]:
        z = np.polyfit(df['Total_Moves'], df[time_col], 1)
        p = np.poly1d(z)
        xp = np.linspace(df['Total_Moves'].min(), df['Total_Moves'].max(), 100)
        plt.plot(xp, p(xp), color=col, linestyle='--', linewidth=1.5, label=lbl)

    plt.xlabel('Total Game Duration (Total Moves)')
    plt.ylabel('Average Turn Execution Latency (Seconds)')
    plt.title(f'Complexity Scaling ({game_title})')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.savefig(f'{game_title.lower()}_iter{iterations}_latency.png', dpi=300)
    plt.close()

if __name__ == "__main__":
    # Ensure these point to your completed high-budget files
    analyze_game_data("csf_hex_11x11_FINAL.csv", "Hex", 5000)
    analyze_game_data("csf_gomoku_15x15_FINAL.csv", "Gomoku", 5000)

         HEX ANALYSIS REPORT (100 GAMES | 5000 ITERATIONS)

--- 1. HEAD-TO-HEAD WIN METRICS ---
UCT Wins:                43 (43.0%)
Thompson Sampling Wins:  57 (57.0%)

First Player (P1) Wins:   53 (53.0%)
Second Player (P2) Wins:  47 (47.0%)
First-Mover Bias (p-value): 0.54851
 -> Verdict: No statistically significant bias.

--- 2. SEARCH BEHAVIOR METRICS ---
UCT Mean Root Entropy:              4.5479
Thompson Sampling Mean Root Entropy: 4.2925
UCT Mean Time per Turn:              63.18 sec
Thompson Sampling Mean Time per Turn: 61.70 sec
Average Moves per Game:              41.0 steps

         GOMOKU ANALYSIS REPORT (100 GAMES | 5000 ITERATIONS)

--- 1. HEAD-TO-HEAD WIN METRICS ---
UCT Wins:                34 (34.0%)
Thompson Sampling Wins:  66 (66.0%)

First Player (P1) Wins:   56 (56.0%)
Second Player (P2) Wins:  44 (44.0%)
First-Mover Bias (p-value): 0.23014
 -> Verdict: No statistically significant bias.

--- 2. SEARCH BEHAVIOR METRICS ---
UCT Mean Root Entropy:              5.30